# Project 13 · BioBERT for Disease Named Entity Recognition

Fine-tune `dmis-lab/biobert-base-cased-v1.1` on NCBI Disease. This run uses the `bert-base-cased` tokenizer configured below.

## 1. Set up the environment

Use a Jupyter kernel with PyTorch, Transformers, Datasets, Evaluate, and seqeval. The BERT baseline notebook lists the dependency setup.

## 2. Load and inspect the dataset

Load the NCBI Disease dataset with its existing training, validation, and test splits. Inspect a training sentence and the three BIO labels: `O`, `B-Disease`, and `I-Disease`.

In [1]:
from datasets import load_dataset

dataset = load_dataset("ncbi/ncbi_disease")

print(dataset)

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 941
    })
})


In [2]:
example = dataset["train"][0]

print("ID:")
print(example["id"])

print("\nTokens:")
print(example["tokens"])

print("\nNER tags:")
print(example["ner_tags"])

ID:
0

Tokens:
['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.']

NER tags:
[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]


In [3]:
label_names = dataset["train"].features["ner_tags"].feature.names

print(label_names)

['O', 'B-Disease', 'I-Disease']


In [4]:
for token, tag in zip(example["tokens"], example["ner_tags"]):
    print(f"{token:20} {label_names[tag]}")

Identification       O
of                   O
APC2                 O
,                    O
a                    O
homologue            O
of                   O
the                  O
adenomatous          B-Disease
polyposis            I-Disease
coli                 I-Disease
tumour               I-Disease
suppressor           O
.                    O


## 3. Tokenize and align labels

Assign the original BIO label to the first subword of each word. Use `-100` for later subwords and special tokens so they are ignored. Inspect one example before applying the same alignment to all splits.

In [6]:
from transformers import AutoTokenizer

model_name = "dmis-lab/biobert-base-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

print(tokenizer)

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [7]:
tokens = example["tokens"]

tokenized = tokenizer(
    tokens,
    is_split_into_words=True
)

bert_tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

print(bert_tokens)

['[CLS]', 'I', '##dent', '##ification', 'of', 'AP', '##C', '##2', ',', 'a', 'ho', '##mo', '##logue', 'of', 'the', 'ad', '##eno', '##mat', '##ous', 'p', '##oly', '##po', '##sis', 'co', '##li', 't', '##umour', 'suppress', '##or', '.', '[SEP]']


In [8]:
word_ids = tokenized.word_ids()

for bert_token, word_id in zip(bert_tokens, word_ids):
    print(f"{bert_token:15} {word_id}")

[CLS]           None
I               0
##dent          0
##ification     0
of              1
AP              2
##C             2
##2             2
,               3
a               4
ho              5
##mo            5
##logue         5
of              6
the             7
ad              8
##eno           8
##mat           8
##ous           8
p               9
##oly           9
##po            9
##sis           9
co              10
##li            10
t               11
##umour         11
suppress        12
##or            12
.               13
[SEP]           None


In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [11]:
test_data = dataset["train"].select([0])

test_result = tokenize_and_align_labels(test_data)

test_tokens = tokenizer.convert_ids_to_tokens(
    test_result["input_ids"][0]
)

test_labels = test_result["labels"][0]

for token, label in zip(test_tokens, test_labels):
    if label == -100:
        label_name = "IGNORE"
    else:
        label_name = label_names[label]

    print(f"{token:15} {label_name}")

[CLS]           IGNORE
I               O
##dent          IGNORE
##ification     IGNORE
of              O
AP              O
##C             IGNORE
##2             IGNORE
,               O
a               O
ho              O
##mo            IGNORE
##logue         IGNORE
of              O
the             O
ad              B-Disease
##eno           IGNORE
##mat           IGNORE
##ous           IGNORE
p               I-Disease
##oly           IGNORE
##po            IGNORE
##sis           IGNORE
co              I-Disease
##li            IGNORE
t               I-Disease
##umour         IGNORE
suppress        O
##or            IGNORE
.               O
[SEP]           IGNORE


In [12]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})


## 4. Train, validate, and evaluate

Fine-tune for three epochs with a learning rate of `2e-5`, a per-device batch size of `8`, and weight decay of `0.01`. Evaluate on the validation split after each epoch. Report entity-level precision, recall, and F1 on the test split, alongside word-level label accuracy.

In [14]:
from transformers import AutoModelForTokenClassification, BertConfig

id2label = {
    0: "O",
    1: "B-Disease",
    2: "I-Disease"
}

label2id = {
    "O": 0,
    "B-Disease": 1,
    "I-Disease": 2
}

config = BertConfig.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    config=config
)

print(model.__class__.__name__)

W1106 14:15:54.032000 21897 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 52462.09it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight         

BertForTokenClassification


In [15]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [16]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./biobert-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [19]:
train_result = trainer.train()

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.034015,0.043695,0.809409,0.852605,0.830446,0.985398
2,0.020259,0.044818,0.818832,0.872935,0.845018,0.986775
3,0.022360,0.052672,0.842491,0.876747,0.859278,0.987234


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.93it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s]
/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.51it/s]


In [20]:
test_results = trainer.evaluate(tokenized_dataset["test"])

print("Precision:", test_results["eval_precision"])
print("Recall:", test_results["eval_recall"])
print("F1:", test_results["eval_f1"])
print("Accuracy:", test_results["eval_accuracy"])

/opt/anaconda3/envs/neuralnet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.022360,0.051354,3,0.842983,0.894792,0.868115,0.985957


Precision: 0.8429833169774289
Recall: 0.8947916666666667
F1: 0.8681152097018697
Accuracy: 0.9859574641792873
